**Задание:**
1. Для датасета животных обучить MLP
2. Использовать `CustomDataset`, `Sampler`, `collate_fn`
3. Сделать предобработку фичей
4. Попробовать `BatchNorm1d`, `Dropout`
5. Подключить для логирования `TensorBoard` и/или `MLflow`
6. Не забыть разделить выборку на train/valid в соотношении 80/20%
7. Получить точность не ниже 65%.  

~8. Сравнить результаты при разных подходах:~  
   ~1) Масштабировать данные StandardScaler'ом + не использовать BatchNorm после Input слоя~  
   ~2) Не Масштабировать данные + использовать BatchNorm после Input слоя~  
   ~3) StandardScaler + BatchNorm~  

In [1]:
!pip install torchmetrics --quiet

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from torchmetrics import Accuracy

## CustomDataset

In [3]:
data = pd.read_csv("animals.csv", sep="\t")
X = data.drop(columns=["target"])
y = data["target"]

In [4]:
y = y.replace("Died", "Euthanasia")
le = LabelEncoder()
y = le.fit_transform(y)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=1/9, stratify=y_train, random_state=42
)

In [6]:
BINARY_FEATURES = [
    "IsDog",
    "HasName",
    "MixColor",
    "TabbyColor",
    "MixBreed",
    "Domestic",
    "Shorthair",
    "Longhair",
    "Breed_Chihuahua Shorthair Mix",
    "Breed_Domestic Medium Hair Mix",
    "Breed_Domestic Shorthair Mix",
    "Breed_German Shepherd Mix",
    "Breed_Labrador Retriever Mix",
    "Breed_Pit Bull Mix",
    "Breed_Rare",
    "SexStatus_Flawed",
    "SexStatus_Intact",
    "SexStatus_Unknown",
]
NUMERIC_FEATURES = [
    "Age",
    "NameLength",
    "NameFreq",
    "ColorFreqAsIs",
    "ColorFreqBase",
    "Year",
    "Day",
]

In [7]:
class CustomDataset(Dataset):
    def __init__(
        self,
        X, y,
        binary_features: list[str],
        numeric_features: list[str],
        category_features: list[str],
    ) -> None:
        X = X.copy()
        self.binary_features = binary_features.copy()
        self.numeric_features = numeric_features.copy()
        self.category_features = category_features.copy()

        # схлопываем дни недели
        weekday_columns = [f"Weekday_{i}" for i in range(7)]
        X["Weekday"] = np.argmax(X[weekday_columns].values, axis=1)
        X.drop(weekday_columns, axis=1, inplace=True)

        # предобработка циклических признаков
        cyclic_features = ["Hour", "Weekday", "Month"]
        for feature, n in zip(cyclic_features, [24, 7, 12]):
            X[f"{feature}_cos"] = np.cos((2 * np.pi / n) * X[feature])
            X[f"{feature}_sin"] = np.sin((2 * np.pi / n) * X[feature])
            self.numeric_features.extend([f"{feature}_cos", f"{feature}_sin"])
        X.drop(columns=cyclic_features, inplace=True)

        # формирование категориального признака
        X["Gender"] = np.argmax(X[["Sex_Female", "Sex_Male", "Sex_Unknown"]].values, axis=1)
        X.drop(columns=["Sex_Female", "Sex_Male", "Sex_Unknown"], inplace=True)
        self.category_features.append("Gender")

        self.columns = X.columns
        self.X = X.values
        self.y = y

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int) -> tuple:
        X_row = self.X[idx, :]
        y_row = self.y[idx]
        X_row_as_dict = {col: torch.tensor(X_row[i]) for i, col in enumerate(self.columns)}
        return X_row_as_dict, y_row

In [8]:
dataset_train = CustomDataset(
    X_train, y_train,
    binary_features=BINARY_FEATURES,
    numeric_features=NUMERIC_FEATURES,
    category_features=[],
)
dataset_valid = CustomDataset(
    X_valid, y_valid,
    binary_features=BINARY_FEATURES,
    numeric_features=NUMERIC_FEATURES,
    category_features=[],
)
dataset_test = CustomDataset(
    X_test, y_test,
    binary_features=BINARY_FEATURES,
    numeric_features=NUMERIC_FEATURES,
    category_features=[],
)

## DataLoader

In [9]:
BATCH_SIZE = 128

In [10]:
dataloader_train = DataLoader(
    dataset=dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
dataloader_valid = DataLoader(dataset=dataset_valid, batch_size=BATCH_SIZE)
dataloader_test = DataLoader(dataset=dataset_test, batch_size=BATCH_SIZE)

## Model

In [11]:
class MLP(torch.nn.Module):
    def __init__(
        self,
        binary_features: list[str],
        numeric_features: list[str],
        category_features: list[str],
        dropout_p: float,
        linear1_size: int,
        linear2_size: int,
        output_size: int,
    ) -> None:
        super().__init__()

        self.binary_features = binary_features
        self.numeric_features = numeric_features
        self.category_features = category_features

        self.embedding_layer = torch.nn.Embedding(num_embeddings=3, embedding_dim=4)

        self.numeric_bn = torch.nn.BatchNorm1d(len(numeric_features))

        self.dropout = torch.nn.Dropout(dropout_p)

        self.linear1 = torch.nn.Linear(
            in_features=len(binary_features)+len(numeric_features)+4,
            out_features=linear1_size,
        )
        torch.nn.init.kaiming_uniform_(self.linear1.weight)
        self.bn1 = torch.nn.BatchNorm1d(linear1_size)

        self.linear2 = torch.nn.Linear(linear1_size, linear2_size)
        torch.nn.init.kaiming_uniform_(self.linear2.weight)
        self.bn2 = torch.nn.BatchNorm1d(linear2_size)

        self.linear3 = torch.nn.Linear(linear2_size, output_size)

    def forward(self, batch):
        binary_tensor = torch.stack([batch[c] for c in self.binary_features], dim=1).float()
        numeric_tensor = torch.stack([batch[c] for c in self.numeric_features], dim=1).float()
        category_tensor = batch[self.category_features[0]].long()

        numeric_tensor = self.numeric_bn(numeric_tensor)

        embedding_output = self.embedding_layer(category_tensor)

        input = torch.cat([binary_tensor, numeric_tensor, embedding_output], dim=1)

        output = self.linear1(input)
        output = self.bn1(output)
        output = torch.relu(output)
        output = self.dropout(output)

        output = self.linear2(output)
        output = self.bn2(output)
        output = torch.relu(output)
        output = self.dropout(output)

        output = self.linear3(output)

        return output

## Train loop

In [12]:
def fit(model, epochs: int, train_loader, valid_loader, optimizer, criterion):
    train_acc = Accuracy(task="multiclass", num_classes=OUTPUT_SIZE)
    valid_acc = Accuracy(task="multiclass", num_classes=OUTPUT_SIZE)
    for epoch in range(1, epochs+1):
        train_loss, train_accuracy = train_epoch(model, train_acc, train_loader, optimizer, criterion)
        valid_loss, valid_accuracy = eval_epoch(model, valid_acc, valid_loader, criterion)
        print(f"{epoch=}, {train_loss=:.3f}, {valid_loss=:.3f}, {train_accuracy=:.3f}, {valid_accuracy=:.3f}")


def train_epoch(model, acc, loader, optimizer, criterion):
    model.train()
    acc.reset()
    total_loss = 0.0
    n = 0
    for features, label in loader:
        optimizer.zero_grad()
        output = model(features)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        acc(output, label)
        total_loss += loss.item()
        n += 1
    return total_loss / n, acc.compute().item()


def eval_epoch(model, acc, loader, criterion):
    model.eval()
    acc.reset()
    total_loss = 0.0
    n = 0
    with torch.no_grad():
        for features, label in loader:
            output = model(features)
            loss = criterion(output, label)

            acc(output, label)
            total_loss += loss.item()
            n += 1
    return total_loss / n, acc.compute().item()

---

In [13]:
DROPOUT_P = 0.3
LINEAR1_SIZE = 128
LINEAR2_SIZE = 64
OUTPUT_SIZE = 4

EPOCHS = 50
LEARNING_RATE = 1e-4

In [14]:
model = MLP(
    binary_features=dataset_train.binary_features,
    numeric_features=dataset_train.numeric_features,
    category_features=dataset_train.category_features,
    dropout_p=DROPOUT_P,
    linear1_size=LINEAR1_SIZE,
    linear2_size=LINEAR2_SIZE,
    output_size=OUTPUT_SIZE,
)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.CrossEntropyLoss()

In [15]:
fit(
    model,
    epochs=EPOCHS,
    train_loader=dataloader_train,
    valid_loader=dataloader_valid,
    optimizer=optimizer,
    criterion=criterion,
)

epoch=1, train_loss=1.327, valid_loss=1.191, train_accuracy=0.369, valid_accuracy=0.501
epoch=2, train_loss=1.150, valid_loss=1.071, train_accuracy=0.506, valid_accuracy=0.583
epoch=3, train_loss=1.071, valid_loss=1.006, train_accuracy=0.552, valid_accuracy=0.594
epoch=4, train_loss=1.022, valid_loss=0.964, train_accuracy=0.576, valid_accuracy=0.611
epoch=5, train_loss=0.991, valid_loss=0.937, train_accuracy=0.588, valid_accuracy=0.618
epoch=6, train_loss=0.969, valid_loss=0.913, train_accuracy=0.601, valid_accuracy=0.626
epoch=7, train_loss=0.953, valid_loss=0.897, train_accuracy=0.605, valid_accuracy=0.633
epoch=8, train_loss=0.941, valid_loss=0.886, train_accuracy=0.612, valid_accuracy=0.639
epoch=9, train_loss=0.934, valid_loss=0.875, train_accuracy=0.614, valid_accuracy=0.645
epoch=10, train_loss=0.921, valid_loss=0.866, train_accuracy=0.621, valid_accuracy=0.649
epoch=11, train_loss=0.911, valid_loss=0.858, train_accuracy=0.622, valid_accuracy=0.651
epoch=12, train_loss=0.904, va